# T2.L3 Demo — нелінійне програмування
Applied model → verification → sensitivity → non-convex multi-start → grid check.

In [ ]:
from pathlib import Path
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT/'src'))
from model import *

## 1. Прикладна нелінійна модель

In [ ]:
baseline = solve_applied(100, (50,50))
baseline, verify_applied_solution(baseline, 100)

## 2. Sensitivity analysis

In [ ]:
sens = pd.DataFrame(applied_sensitivity([60,80,100,120]))
display(sens)
ax=sens.plot(x='total_resource', y='objective', marker='o', legend=False)
ax.set_xlabel('Доступний ресурс R'); ax.set_ylabel('Оптимальне F*'); ax.set_title('Diminishing returns'); plt.show()

## 3. Вплив стартової точки в applied model

In [ ]:
starts=[(10,10),(80,10),(10,80),(50,50)]
pd.DataFrame([{**{'start':str(s)},**solve_applied(100,s)} for s in starts])

## 4. Non-convex multi-start і grid-search verification

In [ ]:
starts_df=pd.read_csv(ROOT/'data'/'nonconvex_starts.csv')
multi=pd.DataFrame(multistart_nonconvex(starts_df[['start_x','start_y']].to_numpy()))
grid=grid_search_nonconvex(0.05)
display(multi)
grid

In [ ]:
xs=np.linspace(-4,4,220); ys=np.linspace(-4,4,220)
X,Y=np.meshgrid(xs,ys)
Z=np.sin(1.7*X)*np.cos(1.3*Y)+0.15*X-0.03*(X*X+Y*Y)
plt.figure(figsize=(7,5)); plt.contourf(X,Y,Z,levels=25)
plt.scatter(multi.x,multi.y,label='local optima'); plt.scatter([grid['x']],[grid['y']],marker='*',s=160,label='grid best')
plt.xlabel('x'); plt.ylabel('y'); plt.title('Non-convex landscape'); plt.legend(); plt.colorbar(); plt.show()

## Висновок
`success=True` не є доказом глобального оптимуму. Для нелінійних задач потрібні feasibility checks, multi-start, sensitivity та незалежні sanity checks.